<a href="https://colab.research.google.com/github/shanmukhyss/flyrankrepo/blob/main/work/notebooks/w03_data_contract.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-04 — Search Intelligence Data Contract

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/shanmukhyss/flyrankrepo/blob/main/work/notebooks/w03_data_contract.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Unit of Analysis

##1. What does one row mean?

One row represents the daily performance metrics for one content page (content_hash_id) belonging to one client (client_hash_id) on a specific reporting date (report_date).

##2. Which table(s) are you using?

I use the fact_content_daily_performance table from the FlyRank internship warehouse.

##3. What time window?

I use March 2026 (2026-03), a mid-panel month, to avoid developing on the final outcome month.

##4. What are you predicting?

I predict the number of daily Google Search Console clicks (gsc_clicks) for a content page.

##5. What do you deliberately exclude?

I exclude rows where GSC or GA4 data is unavailable and do not use the final month (June 2026).

## 2. Fields: feature / label / context / excluded

*Sort every field you plan to touch into these four buckets. Excluded needs a why.*

In [16]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
from google.colab import userdata
import duckdb

HF_TOKEN = userdata.get("HF_TOKEN")

con = duckdb.connect()

con.execute(f"""
CREATE SECRET (
    TYPE huggingface,
    TOKEN '{HF_TOKEN}'
)
""")

In [17]:
rel = "hf://datasets/FlyRank/internship-warehouse"

df = con.sql(f"""
SELECT *
FROM read_parquet('{rel}/fact_content_daily_performance/**/*.parquet')
WHERE YEAR(report_date) = 2026
  AND MONTH(report_date) = 3
""").df()

print(df.shape)
print(df["report_date"].min())
print(df["report_date"].max())

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

(9841378, 31)
2026-03-01 00:00:00
2026-03-31 00:00:00


In [19]:
#grain verification

con.sql(f"""
SELECT
    COUNT(*) AS total_rows,
    COUNT(DISTINCT (report_date, client_hash_id, content_hash_id)) AS unique_rows
FROM read_parquet('{rel}/fact_content_daily_performance/**/*.parquet')
WHERE YEAR(report_date)=2026
AND MONTH(report_date)=3;
""")

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

┌────────────┬─────────────┐
│ total_rows │ unique_rows │
│   int64    │    int64    │
├────────────┼─────────────┤
│    9841378 │     9841378 │
└────────────┴─────────────┘

In [20]:
#Row count & date span

con.sql(f"""
SELECT
    COUNT(*) AS rows,
    MIN(report_date) AS start_date,
    MAX(report_date) AS end_date
FROM read_parquet('{rel}/fact_content_daily_performance/**/*.parquet')
WHERE YEAR(report_date)=2026
AND MONTH(report_date)=3;
""").df()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,rows,start_date,end_date
0,9841378,2026-03-01,2026-03-31


In [21]:
#availability — filter with IS TRUE


con.sql(f"""
SELECT
    COUNT(*) AS total_rows,
    COUNT(*) FILTER (WHERE gsc_data_available IS TRUE) AS gsc_rows,
    COUNT(*) FILTER (WHERE ga4_data_available IS TRUE) AS ga4_rows,
    COUNT(*) FILTER (
        WHERE gsc_data_available IS TRUE
          AND ga4_data_available IS TRUE
    ) AS both_available
FROM read_parquet('{rel}/fact_content_daily_performance/**/*.parquet')
WHERE YEAR(report_date)=2026
AND MONTH(report_date)=3;
""").df()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,total_rows,gsc_rows,ga4_rows,both_available
0,9841378,3611061,413966,364347


## 3. Verify it with queries (grain, counts, missing values, windows)

*Every claim above gets a query cell here. A contract claim without a query next to it is a guess.*

In [22]:
features = con.sql(f"""
SELECT
    report_date,
    client_hash_id,
    content_hash_id,

    gsc_impressions,
    gsc_avg_position,
    ga4_pageviews,
    sessions_organic,
    scroll_events,

    gsc_clicks
FROM read_parquet('{rel}/fact_content_daily_performance/**/*.parquet')
WHERE YEAR(report_date)=2026
AND MONTH(report_date)=3
AND gsc_data_available IS TRUE
AND ga4_data_available IS TRUE;
""").df()

features.head()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,report_date,client_hash_id,content_hash_id,gsc_impressions,gsc_avg_position,ga4_pageviews,sessions_organic,scroll_events,gsc_clicks
0,2026-03-01,client_65de48885f4ef01b,content_5c80451459c29b4a,5,5.400000,1,1,0,0
1,2026-03-01,client_65de48885f4ef01b,content_b1f61fc81b28b2d4,39,5.666667,2,0,0,0
2,2026-03-01,client_65de48885f4ef01b,content_e25ea7297a1dffd3,179,5.156425,2,0,0,0
3,2026-03-01,client_65de48885f4ef01b,content_6b0149a80607dac3,72,7.694444,1,0,0,0
4,2026-03-01,client_65de48885f4ef01b,content_62673eea26c31c17,3282,6.167885,1,1,0,1


In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 4. Data limits

*What can this data never tell you? Unbalanced history, GSC-only early rows, window overlaps.*

In [24]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
print(features.dtypes)


report_date         datetime64[us]
client_hash_id              object
content_hash_id             object
gsc_impressions              int64
gsc_avg_position           float64
ga4_pageviews                int64
sessions_organic             int64
scroll_events                int64
gsc_clicks                   int64
dtype: object


In [26]:
#as we have only 5 numeric features we only take those
X = features[
    [
        "gsc_impressions",
        "gsc_avg_position",
        "ga4_pageviews",
        "sessions_organic",
        "scroll_events"
    ]
]

y = features["gsc_clicks"]
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import r2_score

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42
)

model = RandomForestRegressor(random_state=42)
model.fit(X_train, y_train)

pred = model.predict(X_test)

print("Honest R²:", r2_score(y_test, pred))

Honest R²: 0.8140599903714456


In [27]:
# leakage experiment
X_leak = X.copy()
X_leak["label_copy"] = y

X_train, X_test, y_train, y_test = train_test_split(
    X_leak,
    y,
    test_size=0.2,
    random_state=42
)

model = RandomForestRegressor(random_state=42)
model.fit(X_train, y_train)

pred = model.predict(X_test)

print("Leakage R²:", r2_score(y_test, pred))

Leakage R²: 0.9994902343683154


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.